# EquiSense Modeling: Optuna Tuning + Time-Series CV

Цель: сравнить baseline и бустинг/ансамбли на задаче `next-day direction` с корректной временной валидацией.

**Модели:**
- Logistic Regression (baseline)
- XGBoost + Optuna
- LightGBM + Optuna (или RandomForest + Optuna как fallback)

**Критерии:** ROC-AUC (основной), F1 (дополнительный).

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier
try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False

ROOT = Path.cwd().resolve().parents[0]
BACKEND = ROOT / "backend"
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

from app.features.feature_store import FeatureStore

sns.set_theme(style="whitegrid", context="talk")
optuna.logging.set_verbosity(optuna.logging.WARNING)
print({"lightgbm_available": HAS_LGBM})

In [ ]:
TICKER = "AAPL"
store = FeatureStore()
df = store.build_combined(TICKER).sort_values("date").reset_index(drop=True)

target = (df["returns"].shift(-1) > 0).astype(int)
X = df.drop(columns=["date"]).replace([np.inf, -np.inf], np.nan).fillna(0.0)

valid = ~target.isna()
X = X.loc[valid]
y = target.loc[valid].astype(int)

print("X shape:", X.shape, "| positive rate:", y.mean().round(3))

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

def eval_model(model):
    aucs, f1s = [], []
    for tr, va in tscv.split(X):
        X_tr, X_va = X.iloc[tr], X.iloc[va]
        y_tr, y_va = y.iloc[tr], y.iloc[va]
        model.fit(X_tr, y_tr)
        proba = model.predict_proba(X_va)[:, 1]
        pred = (proba >= 0.5).astype(int)
        aucs.append(roc_auc_score(y_va, proba))
        f1s.append(f1_score(y_va, pred))
    return float(np.mean(aucs)), float(np.mean(f1s))

In [ ]:
baseline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=300, random_state=42))
])
base_auc, base_f1 = eval_model(baseline)
print({"model": "logreg", "auc": round(base_auc, 4), "f1": round(base_f1, 4)})

In [ ]:
def objective_xgb(trial):
    model = XGBClassifier(
        n_estimators=trial.suggest_int("n_estimators", 100, 600),
        max_depth=trial.suggest_int("max_depth", 2, 8),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        random_state=42,
        eval_metric="logloss",
    )
    auc, _ = eval_model(model)
    return auc

study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb, n_trials=35)
best_xgb = XGBClassifier(**study_xgb.best_params, random_state=42, eval_metric="logloss")
xgb_auc, xgb_f1 = eval_model(best_xgb)
print("xgb best", study_xgb.best_params)
print({"model": "xgboost", "auc": round(xgb_auc, 4), "f1": round(xgb_f1, 4)})

In [ ]:
if HAS_LGBM:
    def objective_lgbm(trial):
        model = LGBMClassifier(
            n_estimators=trial.suggest_int("n_estimators", 100, 600),
            num_leaves=trial.suggest_int("num_leaves", 16, 128),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            feature_fraction=trial.suggest_float("feature_fraction", 0.6, 1.0),
            bagging_fraction=trial.suggest_float("bagging_fraction", 0.6, 1.0),
            bagging_freq=trial.suggest_int("bagging_freq", 1, 10),
            random_state=42,
            verbose=-1,
        )
        auc, _ = eval_model(model)
        return auc

    study_lgbm = optuna.create_study(direction="maximize")
    study_lgbm.optimize(objective_lgbm, n_trials=35)
    best_lgbm = LGBMClassifier(**study_lgbm.best_params, random_state=42, verbose=-1)
    lgbm_auc, lgbm_f1 = eval_model(best_lgbm)
    third_model_name = "LightGBM+Optuna"
    print("lgbm best", study_lgbm.best_params)
    print({"model": "lightgbm", "auc": round(lgbm_auc, 4), "f1": round(lgbm_f1, 4)})
else:
    def objective_rf(trial):
        model = RandomForestClassifier(
            n_estimators=trial.suggest_int("n_estimators", 100, 500),
            max_depth=trial.suggest_int("max_depth", 3, 14),
            min_samples_split=trial.suggest_int("min_samples_split", 2, 20),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 8),
            random_state=42,
            n_jobs=-1,
        )
        auc, _ = eval_model(model)
        return auc

    study_lgbm = optuna.create_study(direction="maximize")
    study_lgbm.optimize(objective_rf, n_trials=35)
    best_rf = RandomForestClassifier(**study_lgbm.best_params, random_state=42, n_jobs=-1)
    lgbm_auc, lgbm_f1 = eval_model(best_rf)
    third_model_name = "RandomForest+Optuna"
    print("rf best", study_lgbm.best_params)
    print({"model": "random_forest", "auc": round(lgbm_auc, 4), "f1": round(lgbm_f1, 4)})

In [ ]:
results = pd.DataFrame([
    {"model": "LogReg", "AUC": base_auc, "F1": base_f1},
    {"model": "XGBoost+Optuna", "AUC": xgb_auc, "F1": xgb_f1},
    {"model": third_model_name, "AUC": lgbm_auc, "F1": lgbm_f1},
])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.barplot(data=results, x="model", y="AUC", ax=axes[0, 0], palette="viridis")
axes[0, 0].set_title("AUC by model")
axes[0, 0].tick_params(axis="x", rotation=20)

sns.barplot(data=results, x="model", y="F1", ax=axes[0, 1], palette="mako")
axes[0, 1].set_title("F1 by model")
axes[0, 1].tick_params(axis="x", rotation=20)

xgb_trials = pd.DataFrame({"trial": np.arange(len(study_xgb.trials)), "auc": [t.value for t in study_xgb.trials]})
ens_trials = pd.DataFrame({"trial": np.arange(len(study_lgbm.trials)), "auc": [t.value for t in study_lgbm.trials]})

sns.lineplot(data=xgb_trials, x="trial", y="auc", marker="o", ax=axes[1, 0], label="XGB")
sns.lineplot(data=ens_trials, x="trial", y="auc", marker="o", ax=axes[1, 0], label=third_model_name)
axes[1, 0].set_title("Optuna optimization progress")
axes[1, 0].set_ylabel("validation ROC-AUC")

metrics_long = results.melt(id_vars="model", value_vars=["AUC", "F1"], var_name="metric", value_name="score")
sns.stripplot(data=metrics_long, x="metric", y="score", hue="model", dodge=True, ax=axes[1, 1], palette="tab10")
axes[1, 1].set_title("Metric spread across models")
axes[1, 1].legend(loc="best", fontsize=9)

plt.tight_layout()
results.sort_values("AUC", ascending=False)

## Выводы по моделированию

- Бейзлайн даёт отправную точку, но Optuna-подбор обычно поднимает ROC-AUC.
- Временная кросс-валидация критична: метрики ниже, чем при случайном split, но зато реалистичнее.
- Если `LightGBM` недоступен в окружении, fallback на `RandomForest` сохраняет воспроизводимость эксперимента.
- По итогам стоит выбирать champion-модель по ROC-AUC, а затем проверять стратегический PnL/риски в backtest.

**Следующий шаг:** добавить walk-forward retraining и анализ стабильности метрик по рыночным фазам.